# What does MDM_Id identify?

Test three hypotheses: a canonical product/entity across retailers; a retailer listing; or an ingestion/source row. The name MDM_Id suggests an internal identifier, but does not establish its semantics.

Read both supplied CSVs as strings to preserve identifiers, leading zeros, and scientific-notation text. Keep dataset and zero-based source-row provenance. Examine raw data first, then a transparent structural-screen subset. No source data is changed, and the screen is not a declaration that all retained rows are correct or all excluded rows are invalid.


## Results from this run

- Combined input: **112,634 rows**. The transparent structural screen retains **111,992**, excluding **642** for review.
- Screened rows have **111,992 unique MDM_Ids**, with no repeated IDs or shared IDs between training and target. Raw apparent collisions must be interpreted alongside the malformed/shifted-field examples.
- **0 exact retailer/URL groups** have multiple IDs, versus **807 retailer/SKU groups**. This supports exact-listing/source-record grain more than a unique retailer/SKU identity.
- **158 checksum-valid candidate GTIN groups span retailers**; **158** have multiple MDM_Ids. This weighs against treating MDM_Id as a universal canonical-product ID.
- **Listing identifier versus ingestion-record identifier remains unresolved.** Every retained ID appears once, so this export cannot establish whether IDs persist on reingestion. Timestamp patterns are supporting observations, not proof of allocation semantics.


In [1]:
from pathlib import Path
from urllib.parse import urlsplit, urlunsplit
import pandas as pd
import numpy as np
import re
ROOT=Path.cwd()
if not (ROOT/'data'/'provided').exists():ROOT=ROOT.parent
DATA=ROOT/'data'/'provided'
frames=[]
for dataset in ['Training','Target']:
    frame=pd.read_csv(DATA/f'Selfcare_{dataset}_data (1).csv',dtype='string',low_memory=False)
    frame['_dataset']=dataset;frame['_source_row']=frame.index
    frames.append(frame)
data=pd.concat(frames,ignore_index=True)
source_columns=[c for c in data.columns if not c.startswith('_')]
def clean(series):return series.astype('string').str.strip().replace('',pd.NA)
for source,dest in [('MDM_Id','_id'),('Retailer','_retailer'),('Sku','_sku'),('Upc','_upc'),
                    ('ProductUrl','_url'),('MDM_InsertDateTime','_timestamp')]:data[dest]=clean(data[source])
data['_retailer']=data['_retailer'].str.casefold()
data['_name']=clean(data['ProductName']).str.casefold().str.replace(r'\s+',' ',regex=True)
data['_integer_id']=data['_id'].str.fullmatch(r'\d+',na=False)
data['_numeric_timestamp']=pd.to_numeric(data['_timestamp'],errors='coerce')
# Exploratory Excel-serial-like range, not a confirmed source-system timestamp specification.
data['_plausible_timestamp']=data['_numeric_timestamp'].between(40000,60000).fillna(False)
data['_http_url']=data['_url'].str.match(r'^https?://',case=False,na=False)
data['_screen']=data['_integer_id']&data['_plausible_timestamp']&data['_http_url']
screened=data.loc[data['_screen']].copy()
print('Combined rows:',len(data),'structurally screened rows:',len(screened))


Combined rows: 112634 structurally screened rows: 111992


## 1. Raw completeness, identifier format, duplicates, and train/target overlap

In [2]:
def profile(frame,label):
    present=frame['_id'].dropna()
    counts=present.value_counts()
    return {'view':label,'rows':len(frame),'missing_id':int(frame['_id'].isna().sum()),
            'integer_id_rows':int(frame['_integer_id'].sum()),'unique_nonmissing_ids':present.nunique(),
            'repeated_id_values':int(counts.gt(1).sum()),'rows_in_repeated_ids':int(counts[counts.gt(1)].sum()),
            'extra_rows_beyond_one_per_id':int((counts-1).clip(lower=0).sum())}
profiles=[]
for dataset,frame in data.groupby('_dataset'):profiles.append(profile(frame,'raw '+dataset))
profiles.append(profile(data,'raw combined'))
for dataset,frame in screened.groupby('_dataset'):profiles.append(profile(frame,'screened '+dataset))
profiles.append(profile(screened,'screened combined'))
print(pd.DataFrame(profiles).to_string(index=False))
for label,frame in [('raw',data),('integer ID only',data.loc[data['_integer_id']]),('screened',screened)]:
    ids={d:set(g['_id'].dropna()) for d,g in frame.groupby('_dataset')}
    overlap=ids.get('Training',set())&ids.get('Target',set())
    print(label,'cross-dataset shared IDs:',len(overlap),'first 20:',sorted(overlap)[:20])
print('\nID string length distribution (integer-like only):')
print(data.loc[data['_integer_id'],'_id'].str.len().value_counts().sort_index().to_string())
print('\nFrequent noninteger IDs (text truncated for readability):')
bad=data.loc[data['_id'].notna()&~data['_integer_id'],'_id'].value_counts().head(15).rename_axis('id').reset_index(name='rows')
bad['id']=bad['id'].str.slice(0,120)
print(bad.to_string(index=False))


             view   rows  missing_id  integer_id_rows  unique_nonmissing_ids  repeated_id_values  rows_in_repeated_ids  extra_rows_beyond_one_per_id
       raw Target  28082          31            27962                  28000                   8                    59                            51
     raw Training  84552          66            84216                  84328                  12                   170                           158
     raw combined 112634          97           112178                 112316                  13                   234                           221
  screened Target  27909           0            27909                  27909                   0                     0                             0
screened Training  84083           0            84083                  84083                   0                     0                             0
screened combined 111992           0           111992                 111992                   0          

## 2. Are repeated IDs actually shifted-field artifacts?

Inspect MDM_Id together with its timestamp, product URL, and adjacent fields. Excel-serial-like numbers in the ID column and image URLs in the timestamp column are evidence of column alignment problems, not proof of true ID reuse.

Screen requirements: digit-only ID, numeric timestamp in [40000, 60000], and an HTTP(S) ProductUrl. Report nested sensitivity views so exclusions cannot silently erase collisions. The timestamp range is exploratory and must be replaced with the source schema if available.


In [3]:
repeat_counts=data['_id'].value_counts()
repeated_ids=repeat_counts[repeat_counts.gt(1)].index
collision_rows=data.loc[data['_id'].isin(repeated_ids)]
collision_summary=collision_rows.groupby('_id').agg(rows=('_id','size'),datasets=('_dataset','nunique'),
 retailers=('_retailer','nunique'),urls=('_url','nunique'),timestamps=('_timestamp','nunique'),
 retained=('_screen','sum')).sort_values('rows',ascending=False)
print('Repeated raw IDs (ID text truncated):')
print(collision_summary.reset_index().assign(_id=lambda x:x['_id'].str.slice(0,100)).head(25).to_string(index=False))
print('\nCollision examples across neighboring fields:')
print(collision_rows[['_dataset','_source_row','MDM_Id','MDM_InsertDateTime','ProductUrl','ProductImageUrl','Sku','ProductName']].head(15).to_string(index=False,max_colwidth=100))
print('\nScreen exclusions by dataset (flags overlap):')
print(data.assign(noninteger=~data['_integer_id'],timestamp_flag=~data['_plausible_timestamp'],url_flag=~data['_http_url'],excluded=~data['_screen']).groupby('_dataset')[['noninteger','timestamp_flag','url_flag','excluded']].sum().to_string())
print('\nSensitivity to structural screening:')
for label,mask in [('all rows',pd.Series(True,index=data.index)),('integer ID',data['_integer_id']),
                   ('integer ID + plausible timestamp',data['_integer_id']&data['_plausible_timestamp']),
                   ('integer ID + HTTP URL',data['_integer_id']&data['_http_url']),('all three',data['_screen'])]:
    print(profile(data.loc[mask],label))


Repeated raw IDs (ID text truncated):
        _id  rows  datasets  retailers  urls  timestamps  retained
      44936   109         2          2     8         108         0
      44967    26         2          2     1          26         0
45117.83254    18         2          2     3          18         0
 45027.6581    12         2          1     5          12         0
 45089.7987    12         2          1     4          11         0
45145.83061    12         2          3     1          12         0
44998.35338    10         2          1     2          10         0
45055.70023    10         2          2     3          10         0
45177.76383     8         2          2     0           8         0
          3     7         2          1     5           6         0
          2     5         2          1     4           4         0
          4     3         1          1     3           3         0
          1     2         2          1     2           2         0

Collision examples acro

## 3. Does one MDM_Id span products, retailers, or timestamps?

Audit each ID's number of observed retailers, URLs, retailer/SKU pairs, UPCs, names, and timestamps. In a snapshot where every retained ID appears once, these tests are vacuously one-to-one: that alone cannot establish persistent identity or grain. Reverse mappings in the next sections provide more information.


In [4]:
id_table=screened.groupby('_id').agg(rows=('_id','size'),retailers=('_retailer','nunique'),
 urls=('_url','nunique'),skus=('_sku','nunique'),upcs=('_upc','nunique'),names=('_name','nunique'),timestamps=('_timestamp','nunique'))
print('Retained IDs with multiple observed values:')
print(id_table.gt(1).sum().to_string())
print('Exact duplicate source rows, ignoring provenance:',int(data.duplicated(source_columns).sum()))
print('Screened exact duplicate source rows:',int(screened.duplicated(source_columns).sum()))
print('Screened ID range:',pd.to_numeric(screened['_id']).min(),pd.to_numeric(screened['_id']).max())


Retained IDs with multiple observed values:
rows          0
retailers     0
urls          0
skus          0
upcs          0
names         0
timestamps    0
Exact duplicate source rows, ignoring provenance: 0
Screened exact duplicate source rows: 0
Screened ID range: 284734 2512565


## 4. Can one retailer listing have multiple MDM_Ids?

Compare exact trimmed retailer/URL keys, retailer/SKU keys, and a conservative URL-base key that drops query/fragment and lowercases only the scheme and host. Dropping URL parameters may merge distinct variants, so URL-base matches are candidates only. SKU meaning can also vary by retailer. Names are weaker identity evidence and may omit pack size or variants.


In [5]:
def url_base(value):
    if pd.isna(value):return pd.NA
    try:
        u=urlsplit(value)
        return urlunsplit((u.scheme.lower(),u.netloc.lower(),u.path,'',''))
    except ValueError:return pd.NA
screened['_url_base']=screened['_url'].map(url_base)
relationship_tables={}
relationship_rows=[]
for label,keys in [('retailer + exact URL',['_retailer','_url']),('retailer + URL base',['_retailer','_url_base']),
                   ('retailer + SKU',['_retailer','_sku']),('retailer + normalized name',['_retailer','_name'])]:
    usable=screened.dropna(subset=keys)
    table=usable.groupby(keys).agg(rows=('_id','size'),ids=('_id','nunique'),datasets=('_dataset','nunique'),
      timestamps=('_timestamp','nunique'),names=('_name','nunique'),urls=('_url','nunique'))
    relationship_tables[label]=table
    multi=table['ids'].gt(1)
    relationship_rows.append({'key':label,'eligible_rows':len(usable),'groups':len(table),
     'groups_with_multiple_ids':int(multi.sum()),'rows_in_multi_id_groups':int(table.loc[multi,'rows'].sum()),
     'multi_id_groups_across_datasets':int((multi&table.datasets.gt(1)).sum()),
     'multi_id_groups_with_multiple_timestamps':int((multi&table.timestamps.gt(1)).sum())})
    if multi.any():
        example_keys=table.loc[multi].sort_values('ids',ascending=False).head(5).reset_index()[keys]
        examples=usable.merge(example_keys,on=keys,how='inner')
        print('\nExamples:',label)
        print(examples[['_dataset','_source_row','_id','Retailer','Sku','MDM_InsertDateTime','ProductUrl','ProductName']].head(25).to_string(index=False,max_colwidth=110))
print('\nReverse identity mappings:')
relationship_summary=pd.DataFrame(relationship_rows)
print(relationship_summary.to_string(index=False))



Examples: retailer + URL base
_dataset  _source_row     _id  Retailer        Sku MDM_InsertDateTime                                                                                                     ProductUrl                                                ProductName
Training         8837  818711 Walgreens  400638464         45089.7987 https://www.walgreens.com/store/c/centrum-men-multivitamin-%26-multimineral-supplements-tablets/ID=30041743...        Men Multivitamin & Multimineral Supplements Tablets
Training        14899  523667 Walgreens    6145648         45027.6581 https://www.walgreens.com/store/c/advil-allergy-%26-congestion-relief-coated-tablets/ID=prod6163275-product...                 Allergy & Congestion Relief Coated Tablets
Training        18001 1259386 Walgreens  400638466        45145.83061 https://www.walgreens.com/store/c/centrum-silver-men-50-,-multivitamin-%26-multimineral-supplements-tablets...   Men 50+, Multivitamin & Multimineral Supplements Tablets
Training 

## 5. Is MDM_Id a cross-retailer canonical product identifier?

Use intact 8/12/13/14-digit UPC/GTIN strings and test their check digit. Do not expand scientific-notation UPCs: exported values may already be rounded. Preserve leading zeros; for the candidate GTIN identity key, left-pad checksum-valid values to 14 digits. A valid checksum does not prove a correct UPC or identical package contents. Inspect product names alongside cross-retailer matches.

If plausible same-GTIN products across retailers have different IDs, a universal cross-retailer canonical-product interpretation is poorly supported. It remains possible that the source system uses a narrower entity definition.


In [6]:
def gtin_valid(value):
    if pd.isna(value) or not re.fullmatch(r'(?:\d{8}|\d{12}|\d{13}|\d{14})',value):return False
    if len(set(value))==1:return False
    total=sum(int(d)*(3 if i%2==0 else 1) for i,d in enumerate(reversed(value[:-1])))
    return (10-total%10)%10==int(value[-1])
upc_present=screened['_upc'].notna()
upc_digits=screened['_upc'].str.fullmatch(r'(?:\d{8}|\d{12}|\d{13}|\d{14})',na=False)
valid_gtin=screened['_upc'].map(gtin_valid)
print('Nonmissing UPC rows:',int(upc_present.sum()))
print('Scientific-notation UPC rows:',int(screened['_upc'].str.contains(r'[eE][+-]?\d+',na=False).sum()))
print('Intact supported-length digit UPC rows:',int(upc_digits.sum()))
print('Checksum-valid, nonconstant GTIN rows:',int(valid_gtin.sum()))
gtin_rows=screened.loc[valid_gtin].copy()
gtin_rows['_gtin']=gtin_rows['_upc'].str.zfill(14)
gtin_table=gtin_rows.groupby('_gtin').agg(rows=('_id','size'),ids=('_id','nunique'),retailers=('_retailer','nunique'),
 datasets=('_dataset','nunique'),names=('_name','nunique'))
cross_gtin=gtin_table.loc[gtin_table.retailers.gt(1)]
print('GTIN groups:',len(gtin_table),'groups with multiple IDs:',int(gtin_table.ids.gt(1).sum()))
print('Cross-retailer GTIN groups:',len(cross_gtin),'with multiple IDs:',int(cross_gtin.ids.gt(1).sum()))
print('Cross-dataset GTIN groups:',int(gtin_table.datasets.gt(1).sum()))
examples=gtin_rows.loc[gtin_rows['_gtin'].isin(cross_gtin.sort_values('retailers',ascending=False).head(8).index)]
print(examples[['_gtin','_dataset','_source_row','_id','Retailer','Upc','ProductName']].sort_values(['_gtin','Retailer']).to_string(index=False,max_colwidth=120))


Nonmissing UPC rows: 22186
Scientific-notation UPC rows: 6696
Intact supported-length digit UPC rows: 9593
Checksum-valid, nonconstant GTIN rows: 9593
GTIN groups: 9391 groups with multiple IDs: 192
Cross-retailer GTIN groups: 158 with multiple IDs: 158
Cross-dataset GTIN groups: 81
         _gtin _dataset  _source_row     _id Retailer          Upc                                                                                                              ProductName
00016500576082 Training        60609  587882   Target 016500576082                                                                    The Flintstones Kids Complete Chewable Tablet - 180ct
00016500576082 Training        54911  587056  Walmart 016500576082 Flintstones Vitamins Chewable Kids Vitamins, Complete Multivitamin for Kids and Toddlers with Iron, Calcium, Vitamin ...
00016500592877   Target        23896  467614   Target 016500592877                                                                   Alka-Seltzer Antaci

## 6. Does ID ordering look related to ingestion?

Compare numeric IDs to raw numeric timestamps. For display only, interpret timestamps as Excel-style serial days with origin 1899-12-30; this is a plausible encoding, not a verified schema. Correlation or timestamp batches can be consistent with sequential allocation but cannot distinguish a listing ID from an ingestion-event ID. Proving persistence requires repeated source snapshots or source-system documentation.


In [7]:
time_view=screened.copy()
time_view['_id_number']=pd.to_numeric(time_view['_id'])
time_view['_candidate_datetime']=pd.to_datetime(time_view['_numeric_timestamp'].astype(float),unit='D',origin='1899-12-30',errors='coerce')
print('Candidate timestamp range:',time_view['_candidate_datetime'].min(),'to',time_view['_candidate_datetime'].max())
print('Distinct raw numeric timestamps:',time_view['_numeric_timestamp'].nunique())
print('Spearman-style rank correlation (ID versus numeric timestamp):',time_view['_id_number'].rank().corr(time_view['_numeric_timestamp'].rank()))
batches=time_view.groupby('_timestamp').agg(rows=('_id','size'),ids=('_id','nunique'),retailers=('_retailer','nunique'),
 min_id=('_id_number','min'),max_id=('_id_number','max')).sort_values('rows',ascending=False)
print('Largest timestamp batches:');print(batches.head(15).to_string())
print('\nBy retailer:')
print(time_view.groupby('_retailer').agg(rows=('_id','size'),timestamps=('_timestamp','nunique'),min_id=('_id_number','min'),max_id=('_id_number','max')).to_string())


Candidate timestamp range: 2023-01-10 00:00:00 to 2023-09-08 18:19:54.912000302
Distinct raw numeric timestamps: 12
Spearman-style rank correlation (ID versus numeric timestamp): 0.9576842113281594
Largest timestamp batches:
              rows    ids  retailers   min_id   max_id
_timestamp                                            
44936        48108  48108          5   284734   335230
44967        12284  12284          5   372476   384864
45117.83254  10695  10695          6  1136754  1147495
45145.83061   8048   8048          6  1258178  1266256
44998.35338   7421   7421          6   461853   469316
45027.6581    7234   7234          6   520672   527945
45177.76383   6202   6202          6  2506340  2512565
45089.7987    6063   6063          5   817281   823372
45055.70023   5578   5578          6   584573   590181
45058.43501    211    211          5   680604   680817
44977.33627    135    135          1   390225   390359
44977.3512      13     13          1   390361   390379

By r

## 7. Interpretation and implications

- **Canonical product across retailers:** judge using the same-GTIN examples, not the column name. Multiple MDM_Ids per plausible product contradict a universal canonical-product key in this export.
- **Retailer listing:** one ID per exact retailer/URL is consistent with listing-level grain. Repeated retailer/SKU and normalized URL examples test whether ?listing? really means SKU or exact URL. A single export cannot prove ID stability.
- **Ingestion/source row:** uniqueness and timestamp association are compatible with record allocation, but do not prove that every reingestion generates a new ID. No repeated snapshots means this hypothesis remains unresolved.
- **Data quality:** raw collisions on shifted rows must not be treated as proven source-system ID collisions. Keep provenance, flags, and exclusions reviewable.
- **Practical use:** retain MDM_Id as source traceability metadata. Do not use it alone to deduplicate canonical products, group model evaluation splits, join variants, or infer product attributes. No shared MDM_Ids does not establish product independence between training and target.

**Ask the data owner:** What table and entity does MDM_Id key? Is it unique globally or within a source/retailer? Is it stable across updates and reingestion? Can one SKU/GTIN have multiple MDM_Ids? What does MDM_InsertDateTime record, and how was it exported? Request repeated snapshots or an entity/listing mapping to distinguish the remaining hypotheses.
